In [1]:
! pip install optuna
! pip install mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 28.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66

# BigMartSales

Imports and Global Configs

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor

import optuna
import mlflow
import mlflow.sklearn

RANDOM_STATE = 42
N_SPLITS = 5

mlflow.set_experiment("BigMartSales_Prediction")


2026/02/12 15:53:36 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/02/12 15:53:36 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/02/12 15:53:36 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/02/12 15:53:36 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/02/12 15:53:36 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/02/12 15:53:36 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/02/12 15:53:36 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/02/12 15:53:36 INFO mlflow.store.db.utils: Updating database tables
2026/02/12 15:53:36 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/02/12 15:53:36 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/02/12 15:53:37 INFO alembic.runtime.migration: Running upgrade  -> 451aebb31d03, add metric step
2026/02/12 15:5

<Experiment: artifact_location='/content/mlruns/1', creation_time=1770911618453, experiment_id='1', last_update_time=1770911618453, lifecycle_stage='active', name='BigMartSales_Prediction', tags={}>

Data Loading

In [2]:
train = pd.read_csv("/content/train_v9rqX0R.csv")
test = pd.read_csv("/content/test_AbJTz2l.csv")

test_ids = test[["Item_Identifier", "Outlet_Identifier"]].copy()

print(train.shape, test.shape)

(8523, 12) (5681, 11)


Feature Engineering Pipeline

In [3]:
def feature_engineering(train, test):

    train = train.copy()
    test = test.copy()

    # Handling Missing Item Weight
    item_weight_median = train.groupby("Item_Type")["Item_Weight"].median()

    for df in [train, test]:
        df["Item_Weight"] = df.apply(
            lambda x: item_weight_median[x["Item_Type"]]
            if pd.isnull(x["Item_Weight"]) else x["Item_Weight"],
            axis=1
        )

    # Visibility Handling

    train.loc[train["Item_Visibility"] == 0, "Item_Visibility"] = np.nan
    test.loc[test["Item_Visibility"] == 0, "Item_Visibility"] = np.nan

    visibility_median = train.groupby("Item_Type")["Item_Visibility"].median()

    for df in [train, test]:
        df["Item_Visibility"] = df.apply(
            lambda x: visibility_median[x["Item_Type"]]
            if pd.isnull(x["Item_Visibility"]) else x["Item_Visibility"],
            axis=1
        )

    # Fat Content Normalize + Binary Encoding

    mapping = {
        "lf": "Low Fat",
        "low fat": "Low Fat",
        "reg": "Regular",
        "regular": "Regular"
    }

    for df in [train, test]:
        df["Item_Fat_Content"] = (
            df["Item_Fat_Content"]
            .astype(str)
            .str.strip()
            .str.lower()
            .replace(mapping)
        )

    fat_encoding = {"Low Fat": 1, "Regular": 0}

    for df in [train, test]:
        df["Item_Fat_Content"] = df["Item_Fat_Content"].replace(fat_encoding)

    # Outlet Age Generation

    CURRENT_YEAR = 2013
    for df in [train, test]:
        df["Outlet_Age"] = CURRENT_YEAR - df["Outlet_Establishment_Year"]

    train.drop(columns=["Outlet_Establishment_Year"], inplace=True)
    test.drop(columns=["Outlet_Establishment_Year"], inplace=True)


    # Visibility Ratio

    visibility_mean = train.groupby("Item_Type")["Item_Visibility"].transform("mean")
    train["Visibility_Ratio"] = train["Item_Visibility"] / visibility_mean

    visibility_map = train.groupby("Item_Type")["Item_Visibility"].mean()
    test["Visibility_Ratio"] = test["Item_Visibility"] / test["Item_Type"].map(visibility_map)


    # MRP Binning
    bins = [0, 70, 140, 210, 300]

    train["MRP_Bin"] = pd.cut(train["Item_MRP"], bins=bins, labels=False)
    test["MRP_Bin"] = pd.cut(test["Item_MRP"], bins=bins, labels=False)

    # Interaction Feature

    train["Item_Outlet_Type"] = train["Item_Type"] + "_" + train["Outlet_Type"]
    test["Item_Outlet_Type"] = test["Item_Type"] + "_" + test["Outlet_Type"]

    return train, test


Kfold Encoding

In [4]:
def kfold_target_encoding(train_df, test_df, column, target, n_splits=5):

    train_encoded = pd.Series(index=train_df.index, dtype=float)
    global_mean = train_df[target].mean()

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

    for train_idx, val_idx in kf.split(train_df):
        fold_train = train_df.iloc[train_idx]
        fold_val = train_df.iloc[val_idx]

        means = fold_train.groupby(column)[target].mean()
        train_encoded.iloc[val_idx] = fold_val[column].map(means)

    train_encoded.fillna(global_mean, inplace=True)

    full_means = train_df.groupby(column)[target].mean()
    test_encoded = test_df[column].map(full_means)
    test_encoded.fillna(global_mean, inplace=True)

    return train_encoded, test_encoded


Prepare Modeling Data

In [5]:
def build_dataset(train, test):

    train, test = feature_engineering(train, test)

    # Target encoding
    train["Item_TE"], test["Item_TE"] = kfold_target_encoding(
        train, test, "Item_Identifier", "Item_Outlet_Sales"
    )

    train["Outlet_TE"], test["Outlet_TE"] = kfold_target_encoding(
        train, test, "Outlet_Identifier", "Item_Outlet_Sales"
    )

    # Drop high-card columns
    train.drop(columns=["Item_Identifier", "Outlet_Identifier"], inplace=True)
    test.drop(columns=["Item_Identifier", "Outlet_Identifier"], inplace=True)

    # One hot encode remaining categoricals
    train = pd.get_dummies(train, drop_first=True)
    test = pd.get_dummies(test, drop_first=True)

    train, test = train.align(test, join="left", axis=1, fill_value=0)

    y = train["Item_Outlet_Sales"]
    X = train.drop("Item_Outlet_Sales", axis=1)

    return X, y, test


In [6]:
X, y, test_processed = build_dataset(train, test)


In [7]:
y_log = np.log1p(y)


Model Evaluation Utility

In [8]:
def evaluate_model_log(model, X, y):

    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    scores = []

    for train_idx, val_idx in kf.split(X):

        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        y_tr_log = np.log1p(y_tr)

        model.fit(X_tr, y_tr_log)

        preds_log = model.predict(X_val)
        preds = np.expm1(preds_log)

        rmse = np.sqrt(mean_squared_error(y_val, preds))
        scores.append(rmse)

    return np.mean(scores)



Baseline MLflow Run

In [9]:
with mlflow.start_run(run_name="GB_LogTarget_Baseline"):

    model = GradientBoostingRegressor(
        n_estimators=600,
        learning_rate=0.03,
        max_depth=4,
        min_samples_leaf=5,
        subsample=0.8,
        random_state=RANDOM_STATE
    )

    cv_rmse = evaluate_model_log(model, X, y)

    mlflow.log_params(model.get_params())
    mlflow.log_metric("cv_rmse", cv_rmse)

    print("Log Target CV RMSE:", cv_rmse)


Log Target CV RMSE: 1116.9653106128328


Optuna + MLflow Integration

In [10]:
def objective_log(trial):

    with mlflow.start_run(nested=True):

        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
            "max_depth": trial.suggest_int("max_depth", 2, 6),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "random_state": RANDOM_STATE
        }

        model = GradientBoostingRegressor(**params)

        cv_rmse = evaluate_model_log(model, X, y)

        mlflow.log_params(params)
        mlflow.log_metric("cv_rmse", cv_rmse)

        return cv_rmse


study_log = optuna.create_study(direction="minimize")
study_log.optimize(objective_log, n_trials=50)

print("Best Log CV:", study_log.best_value)
print("Best Log Params:", study_log.best_params)



[I 2026-02-12 16:09:44,575] A new study created in memory with name: no-name-04bfbf64-e290-47a3-9433-e4f716ec179c
[I 2026-02-12 16:10:17,962] Trial 0 finished with value: 1124.8057594263216 and parameters: {'n_estimators': 218, 'learning_rate': 0.09991114274629398, 'max_depth': 4, 'min_samples_leaf': 14, 'subsample': 0.8998213719773049}. Best is trial 0 with value: 1124.8057594263216.
[I 2026-02-12 16:11:20,562] Trial 1 finished with value: 1112.9952792768313 and parameters: {'n_estimators': 750, 'learning_rate': 0.03355073423870846, 'max_depth': 2, 'min_samples_leaf': 8, 'subsample': 0.9407718015698252}. Best is trial 1 with value: 1112.9952792768313.
[I 2026-02-12 16:12:02,264] Trial 2 finished with value: 1118.0754390955258 and parameters: {'n_estimators': 235, 'learning_rate': 0.020118540799466136, 'max_depth': 5, 'min_samples_leaf': 2, 'subsample': 0.8470665648349818}. Best is trial 1 with value: 1112.9952792768313.
[I 2026-02-12 16:12:36,963] Trial 3 finished with value: 1116.277

Best Log CV: 1108.9455969363994
Best Log Params: {'n_estimators': 448, 'learning_rate': 0.022247845544229716, 'max_depth': 3, 'min_samples_leaf': 12, 'subsample': 0.7589360609060707}


In [11]:
test_processed = test_processed[X.columns]


In [12]:
best_model_log = GradientBoostingRegressor(
    **study_log.best_params,
    random_state=RANDOM_STATE
)

# Train on full log target
best_model_log.fit(X, np.log1p(y))

# Align test columns
test_processed = test_processed.reindex(columns=X.columns, fill_value=0)

# Predict
pred_log = best_model_log.predict(test_processed)
predictions = np.expm1(pred_log)

submission = test_ids.copy()
submission["Item_Outlet_Sales"] = predictions

submission.to_csv("submission_log_v5.csv", index=False)



In [13]:
with mlflow.start_run(run_name="Leaderboard_Log"):
    mlflow.log_metric("leaderboard_score", 1157)


In [14]:
mlflow.search_runs()

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.leaderboard_score,metrics.cv_rmse,params.random_state,params.learning_rate,...,params.min_samples_split,params.max_leaf_nodes,params.criterion,params.min_impurity_decrease,params.loss,params.max_features,tags.mlflow.source.name,tags.mlflow.user,tags.mlflow.source.type,tags.mlflow.runName
0,53d7114326bb46a9ba5ababd321bfffc,1,FINISHED,/content/mlruns/1/53d7114326bb46a9ba5ababd321b...,2026-02-12 16:57:38.936000+00:00,2026-02-12 16:57:38.967000+00:00,1157.0,NaN,None,None,...,None,None,None,None,None,None,fileId=1gFGzTOE4wTo6LhuahWfCOstn_Tihi9QJ,root,NOTEBOOK,Leaderboard_Log
1,145ae6735cc84903b021d3429e83f04a,1,FINISHED,/content/mlruns/1/145ae6735cc84903b021d3429e83...,2026-02-12 16:52:13.568000+00:00,2026-02-12 16:53:20.737000+00:00,NaN,1113.279890,42,0.02311481637847976,...,None,None,None,None,None,None,fileId=1gFGzTOE4wTo6LhuahWfCOstn_Tihi9QJ,root,NOTEBOOK,wistful-wren-771
2,13ab528903954057b46073148d1e274f,1,FINISHED,/content/mlruns/1/13ab528903954057b46073148d1e...,2026-02-12 16:51:28.757000+00:00,2026-02-12 16:52:13.556000+00:00,NaN,1108.945597,42,0.022247845544229716,...,None,None,None,None,None,None,fileId=1gFGzTOE4wTo6LhuahWfCOstn_Tihi9QJ,root,NOTEBOOK,delicate-bat-188
3,bb8885054e464049a4d1fbd196625795,1,FINISHED,/content/mlruns/1/bb8885054e464049a4d1fbd19662...,2026-02-12 16:50:58.225000+00:00,2026-02-12 16:51:28.745000+00:00,NaN,1109.787682,42,0.020430518979123503,...,None,None,None,None,None,None,fileId=1gFGzTOE4wTo6LhuahWfCOstn_Tihi9QJ,root,NOTEBOOK,defiant-tern-991
4,d42a3eca658b4b08b010269db56d8a56,1,FINISHED,/content/mlruns/1/d42a3eca658b4b08b010269db56d...,2026-02-12 16:49:34.199000+00:00,2026-02-12 16:50:58.209000+00:00,NaN,1110.026921,42,0.015136447436933603,...,None,None,None,None,None,None,fileId=1gFGzTOE4wTo6LhuahWfCOstn_Tihi9QJ,root,NOTEBOOK,overjoyed-lark-996
5,47b666df1d3c49a18a98af2fa20cd1ce,1,FINISHED,/content/mlruns/1/47b666df1d3c49a18a98af2fa20c...,2026-02-12 16:48:43.312000+00:00,2026-02-12 16:49:34.186000+00:00,NaN,1109.451864,42,0.020627221294557015,...,None,None,None,None,None,None,fileId=1gFGzTOE4wTo6LhuahWfCOstn_Tihi9QJ,root,NOTEBOOK,clean-moose-693
6,f6f3d118e85c4841afa301b136c18460,1,FINISHED,/content/mlruns/1/f6f3d118e85c4841afa301b136c1...,2026-02-12 16:48:23.334000+00:00,2026-02-12 16:48:43.301000+00:00,NaN,1109.887439,42,0.031295605608769844,...,None,None,None,None,None,None,fileId=1gFGzTOE4wTo6LhuahWfCOstn_Tihi9QJ,root,NOTEBOOK,peaceful-snipe-296
7,55be870751dc4eec84aea52a2397f486,1,FINISHED,/content/mlruns/1/55be870751dc4eec84aea52a2397...,2026-02-12 16:46:36.553000+00:00,2026-02-12 16:48:23.321000+00:00,NaN,1134.043433,42,0.04725189309539223,...,None,None,None,None,None,None,fileId=1gFGzTOE4wTo6LhuahWfCOstn_Tihi9QJ,root,NOTEBOOK,nosy-mink-398
8,92097cd139124f4092a0d527ad4e1bc3,1,FINISHED,/content/mlruns/1/92097cd139124f4092a0d527ad4e...,2026-02-12 16:45:52.263000+00:00,2026-02-12 16:46:36.540000+00:00,NaN,1112.189498,42,0.03600797296436142,...,None,None,None,None,None,None,fileId=1gFGzTOE4wTo6LhuahWfCOstn_Tihi9QJ,root,NOTEBOOK,angry-midge-106
9,0154bcd9f91a48cab33bc01bb6e7a170,1,FINISHED,/content/mlruns/1/0154bcd9f91a48cab33bc01bb6e7...,2026-02-12 16:45:05.601000+00:00,2026-02-12 16:45:52.250000+00:00,NaN,1109.869304,42,0.027158710414660732,...,None,None,None,None,None,None,fileId=1gFGzTOE4wTo6LhuahWfCOstn_Tihi9QJ,root,NOTEBOOK,bemused-bee-615
